In [ ]:
import zipfile
import os

# Chemin du fichier ZIP
zip_path = 'amhcd-data-64.zip'
# Dossier de destination
extract_folder = '/content'
os.makedirs(extract_folder, exist_ok=True)

# Extraction
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

print(f"Dossier extrait dans : {extract_folder}")

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import cv2
from pathlib import Path

class TifinaghDataLoader:
    """Data loader for Tifinagh characters dataset"""

    def __init__(self, data_dir, target_size=(32, 32)):
        self.data_dir = data_dir
        self.target_size = target_size
        self.label_encoder = LabelEncoder()

    def load_dataset(self, test_size=0.2, validation_size=0.1):
        """Load and preprocess the Tifinagh dataset"""
        print(f"Loading dataset from: {self.data_dir}")

        images = []
        labels = []
        class_names = []

        # Get all class directories
        class_dirs = [d for d in os.listdir(self.data_dir)
                     if os.path.isdir(os.path.join(self.data_dir, d))]
        class_dirs.sort()

        print(f"Found {len(class_dirs)} classes: {class_dirs}")

        # Load images from each class
        for class_name in class_dirs:
            class_path = os.path.join(self.data_dir, class_name)
            class_images = []

            # Get all image files in the class directory
            image_files = []
            for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tiff']:
                image_files.extend(Path(class_path).glob(ext))
                image_files.extend(Path(class_path).glob(ext.upper()))

            print(f"Class '{class_name}': {len(image_files)} images")

            for image_path in image_files:
                try:
                    # Load and preprocess image
                    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
                    if image is not None:
                        # Resize to target size
                        image = cv2.resize(image, self.target_size)
                        # Normalize to [0, 1]
                        image = image.astype(np.float32) / 255.0

                        images.append(image)
                        labels.append(class_name)

                except Exception as e:
                    print(f"Error loading {image_path}: {e}")
                    continue

        if not images:
            raise ValueError("No images found in the dataset!")

        # Convert to numpy arrays
        X = np.array(images)
        y = np.array(labels)

        # Add channel dimension for grayscale images
        X = np.expand_dims(X, axis=-1)

        # Encode labels
        y_encoded = self.label_encoder.fit_transform(y)

        print(f"Dataset loaded: {X.shape[0]} images, {len(np.unique(y_encoded))} classes")
        print(f"Image shape: {X.shape[1:]}")

        # Split into train/test
        X_train, X_test, y_train, y_test = train_test_split(
            X, y_encoded, test_size=test_size, stratify=y_encoded, random_state=42
        )

        # Split train into train/validation
        X_train, X_val, y_train, y_val = train_test_split(
            X_train, y_train, test_size=validation_size, stratify=y_train, random_state=42
        )

        # Convert labels to categorical
        num_classes = len(np.unique(y_encoded))
        y_train = keras.utils.to_categorical(y_train, num_classes)
        y_val = keras.utils.to_categorical(y_val, num_classes)
        y_test = keras.utils.to_categorical(y_test, num_classes)

        print(f"Train set: {X_train.shape[0]} samples")
        print(f"Validation set: {X_val.shape[0]} samples")
        print(f"Test set: {X_test.shape[0]} samples")

        return (X_train, y_train), (X_val, y_val), (X_test, y_test), num_classes

    def get_class_names(self):
        """Get class names"""
        return self.label_encoder.classes_

    def plot_sample_images(self, X, y, class_names, num_samples=10):
        """Plot sample images from the dataset"""
        fig, axes = plt.subplots(2, 5, figsize=(15, 6))
        axes = axes.ravel()

        for i in range(min(num_samples, len(X))):
            axes[i].imshow(X[i].squeeze(), cmap='gray')
            class_idx = np.argmax(y[i])
            axes[i].set_title(f'{class_names[class_idx]}')
            axes[i].axis('off')

        plt.tight_layout()
        plt.show()

class LeNet5Model:
    """LeNet-5 CNN Architecture for Tifinagh Character Recognition"""

    def __init__(self, num_classes, input_shape=(32, 32, 1)):
        self.num_classes = num_classes
        self.input_shape = input_shape
        self.model = None
        self.history = None

    def build_model(self, activation='tanh'):
        """
        Build LeNet-5 architecture:
        - Input: 32?32 grayscale image
        - C1: 6 filters 5?5 ? 28?28?6
        - S2: Average pooling 2?2 ? 14?14?6
        - C3: 16 filters 5?5 ? 10?10?16
        - S4: Average pooling 2?2 ? 5?5?16
        - C5: Fully connected 120 neurons
        - F6: Fully connected 84 neurons
        - Output: num_classes neurons
        """

        model = keras.Sequential([
            # Input layer
            layers.Input(shape=self.input_shape),

            # C1: Convolution layer - 6 filters of 5?5
            layers.Conv2D(6, (5, 5), activation=activation, padding='valid', name='C1'),
            # Output: 28?28?6

            # S2: Average pooling 2?2
            layers.AveragePooling2D((2, 2), strides=2, name='S2'),
            # Output: 14?14?6

            # C3: Convolution layer - 16 filters of 5?5
            layers.Conv2D(16, (5, 5), activation=activation, padding='valid', name='C3'),
            # Output: 10?10?16

            # S4: Average pooling 2?2
            layers.AveragePooling2D((2, 2), strides=2, name='S4'),
            # Output: 5?5?16

            # Flatten for fully connected layers
            layers.Flatten(),

            # C5: Fully connected layer - 120 neurons
            layers.Dense(120, activation=activation, name='C5'),

            # F6: Fully connected layer - 84 neurons
            layers.Dense(84, activation=activation, name='F6'),

            # Output layer - num_classes neurons
            layers.Dense(self.num_classes, activation='softmax', name='Output')
        ])

        self.model = model
        return model

    def compile_model(self, optimizer='adam', learning_rate=0.001):
        """Compile the model"""
        if optimizer == 'adam':
            opt = keras.optimizers.Adam(learning_rate=learning_rate)
        elif optimizer == 'sgd':
            opt = keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
        else:
            opt = optimizer

        self.model.compile(
            optimizer=opt,
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

    def train(self, X_train, y_train, X_val, y_val, epochs=50, batch_size=32):
        """Train the model"""

        # Callbacks
        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=10,
                restore_best_weights=True
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=1e-7
            )
        ]

        # Train the model
        self.history = self.model.fit(
            X_train, y_train,
            batch_size=batch_size,
            epochs=epochs,
            validation_data=(X_val, y_val),
            callbacks=callbacks,
            verbose=1
        )

        return self.history

    def evaluate(self, X_test, y_test):
        """Evaluate the model"""
        return self.model.evaluate(X_test, y_test, verbose=0)

    def predict(self, X):
        """Make predictions"""
        return self.model.predict(X)

    def plot_training_history(self):
        """Plot training history"""
        if self.history is None:
            print("No training history available!")
            return

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        # Plot training & validation accuracy
        ax1.plot(self.history.history['accuracy'], label='Training Accuracy')
        ax1.plot(self.history.history['val_accuracy'], label='Validation Accuracy')
        ax1.set_title('Model Accuracy')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Accuracy')
        ax1.legend()
        ax1.grid(True)

        # Plot training & validation loss
        ax2.plot(self.history.history['loss'], label='Training Loss')
        ax2.plot(self.history.history['val_loss'], label='Validation Loss')
        ax2.set_title('Model Loss')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()
        plt.show()

    def save_model(self, filepath):
        """Save the trained model"""
        self.model.save(filepath)
        print(f"Model saved to {filepath}")

    def load_model(self, filepath):
        """Load a trained model"""
        self.model = keras.models.load_model(filepath)
        print(f"Model loaded from {filepath}")

    def visualize_filters(self):
        """Visualize convolutional filters"""
        for layer in self.model.layers:
            if isinstance(layer, layers.Conv2D):
                print(f"\nVisualizing filters for layer: {layer.name}")
                filters, biases = layer.get_weights()

                # Normalize filter values to 0-1 for visualization
                f_min, f_max = filters.min(), filters.max()
                filters = (filters - f_min) / (f_max - f_min)

                # Plot first few filters
                n_filters = min(6, filters.shape[3])  # Only show first 6 filters
                fig, axes = plt.subplots(1, n_filters, figsize=(15, 3))

                for i in range(n_filters):
                    ax = axes[i] if n_filters > 1 else axes
                    # Get the filter (for grayscale input, only 1 channel)
                    f = filters[:, :, 0, i]
                    ax.imshow(f, cmap='gray')
                    ax.set_title(f'Filter {i+1}')
                    ax.axis('off')

                plt.suptitle(f'Filters - {layer.name}')
                plt.show()

    def visualize_activations(self, test_image, layer_names=None):
        """Visualize layer activations for a test image"""
        if layer_names is None:
            layer_names = [layer.name for layer in self.model.layers
                          if isinstance(layer, (layers.Conv2D, layers.AveragePooling2D))]

        # Create a model that will return these outputs
        outputs = [self.model.get_layer(name).output for name in layer_names]
        activation_model = keras.models.Model(inputs=self.model.input, outputs=outputs)

        # Get activations
        test_image = np.expand_dims(test_image, axis=0)  # Add batch dimension
        activations = activation_model.predict(test_image)

        # Plot the activations
        for layer_name, layer_activation in zip(layer_names, activations):
            print(f"\nVisualizing outputs for layer: {layer_name}")
            print(f"Activation shape: {layer_activation.shape}")

            # For Conv/Pooling layers
            if len(layer_activation.shape) == 4:
                n_features = min(6, layer_activation.shape[-1])  # Only show first 6 features
                fig, axes = plt.subplots(1, n_features, figsize=(15, 3))

                for i in range(n_features):
                    ax = axes[i] if n_features > 1 else axes
                    ax.imshow(layer_activation[0, :, :, i], cmap='viridis')
                    ax.set_title(f'Feature {i+1}')
                    ax.axis('off')

                plt.suptitle(f'Feature Maps - {layer_name}')
                plt.show()

def main():
    """Main training pipeline"""

    # Dataset configuration
    data_dir = os.path.join(os.getcwd(), 'amhcd-data-64/tifinagh-images/')
    print(f"Dataset directory: {data_dir}")
    print(f"Current working directory: {os.getcwd()}")

    # Check if dataset exists
    if not os.path.exists(data_dir):
        print(f"Dataset directory not found: {data_dir}")
        print("Please make sure the dataset is in the correct location.")
        return

    # Load dataset
    data_loader = TifinaghDataLoader(data_dir, target_size=(32, 32))

    try:
        (X_train, y_train), (X_val, y_val), (X_test, y_test), num_classes = data_loader.load_dataset()
        class_names = data_loader.get_class_names()

        print(f"\nDataset Summary:")
        print(f"Number of classes: {num_classes}")
        print(f"Class names: {class_names}")

        # Plot sample images
        print("\nDisplaying sample images...")
        data_loader.plot_sample_images(X_train, y_train, class_names)

        # Create and build model
        print(f"\nBuilding LeNet-5 model for {num_classes} classes...")
        lenet = LeNet5Model(num_classes=num_classes)

        # Build with tanh activation (original LeNet-5)
        model = lenet.build_model(activation='tanh')

        # Print model architecture
        print("\nModel Architecture:")
        model.summary()

        # Compile model
        lenet.compile_model(optimizer='adam', learning_rate=0.001)

        # Train model
        print("\nStarting training...")
        history = lenet.train(
            X_train, y_train,
            X_val, y_val,
            epochs=50,
            batch_size=32
        )

        # Plot training history
        lenet.plot_training_history()

        # Evaluate on test set
        print("\nEvaluating on test set...")
        test_loss, test_accuracy = lenet.evaluate(X_test, y_test)
        print(f"Test Accuracy: {test_accuracy:.4f}")
        print(f"Test Loss: {test_loss:.4f}")

        # Visualize convolutional filters
        print("\nVisualizing convolutional filters...")
        lenet.visualize_filters()

        # Visualize layer activations for a test image
        test_sample_idx = 0  # You can change this index
        test_image = X_test[test_sample_idx]
        true_label = class_names[np.argmax(y_test[test_sample_idx])]

        print(f"\nVisualizing layer outputs for test image (class: {true_label})...")
        plt.imshow(test_image.squeeze(), cmap='gray')
        plt.title(f"Test Image - Class: {true_label}")
        plt.axis('off')
        plt.show()

        # Visualize activations for all conv/pool layers
        lenet.visualize_activations(test_image)

        # You can also specify particular layers
        # lenet.visualize_activations(test_image, layer_names=['C1', 'S2', 'C3', 'S4'])

        # Save model
        model_path = 'lenet5_tifinagh_model.h5'
        lenet.save_model(model_path)

        # Make predictions on test set
        print("\nMaking predictions on test samples...")
        predictions = lenet.predict(X_test[:10])

        # Show predictions
        fig, axes = plt.subplots(2, 5, figsize=(15, 6))
        axes = axes.ravel()

        for i in range(10):
            axes[i].imshow(X_test[i].squeeze(), cmap='gray')
            pred_class = np.argmax(predictions[i])
            true_class = np.argmax(y_test[i])

            title = f'Pred: {class_names[pred_class]}\nTrue: {class_names[true_class]}'
            color = 'green' if pred_class == true_class else 'red'
            axes[i].set_title(title, color=color)
            axes[i].axis('off')

        plt.tight_layout()
        plt.show()

        print(f"\nTraining completed successfully!")
        print(f"Model saved as: {model_path}")

    except Exception as e:
        print(f"Error during training: {e}")
        print("Please check your dataset structure and file paths.")

if __name__ == "__main__":
    main()